# Sequences: recurrent networks, and the length they stop working at

MichAl Academy, unit 3.10.

Two tasks run through this notebook, both made here rather than downloaded,
because each is built to isolate one thing.

**The order task.** A run of filler with one `A` and one `B` somewhere in it.
The label is which came first. Both classes contain exactly one `A` and one `B`,
so anything that counts tokens is at chance *by construction* and the only way
to answer is to use the order.

**The recall task.** The first token is one of eight, everything after it is
random filler, and the label is that first token. Chance is 0.125. Nothing about
this task is hard except the distance between the evidence and the answer.


In [ ]:
import time
import warnings

import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from torch import nn

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

CLASSES, FILLER = 8, 10          # recall task: tokens 1-8 are labels, 9-18 filler


def order_task(n, length, seed):
    rng = np.random.default_rng(seed)
    X = np.zeros((n, length), dtype=np.int64)
    y = np.zeros(n, dtype=np.int64)
    for i in range(n):
        a, b = rng.choice(length, 2, replace=False)
        X[i, a], X[i, b] = 1, 2
        y[i] = 1 if a < b else 0
    return X, y


def recall_task(n, length, seed):
    rng = np.random.default_rng(seed)
    X = rng.integers(CLASSES + 1, CLASSES + 1 + FILLER,
                     size=(n, length)).astype(np.int64)
    y = rng.integers(0, CLASSES, n)
    X[:, 0] = y + 1
    return X, y


X10, y10 = order_task(4000, 10, 0)
print("one order-task sequence, length 10:", X10[0], "label", y10[0])
print("0 is filler, 1 is A, 2 is B. The label is 1 when A came first.")


## Why a counting model cannot do this

The two classes have identical token counts. Anything reading a bag of tokens
sees the same input for both answers.


In [ ]:
X_test10, y_test10 = order_task(1000, 10, 1)
X20, y20 = order_task(1000, 20, 2)

def token_counts(X):
    return np.stack([(X == t).sum(axis=1) for t in (0, 1, 2)], axis=1)

counter = LogisticRegression(max_iter=1000).fit(token_counts(X10), y10)
print(f"logistic regression on token counts: {counter.score(token_counts(X_test10), y_test10):.4f}")
print("chance is 0.5")


## Why a dense network cannot do this either

A dense network can solve it, perfectly, at the length it was built for. Its
first layer has a fixed number of inputs, so a longer sequence is not a harder
problem for it. It is not a problem it can accept at all.


In [ ]:
class OneHot(nn.Module):
    def __init__(self, inner):
        super().__init__()
        self.inner = inner
    def forward(self, x):
        return self.inner(torch.nn.functional.one_hot(x, 3).float())


def train(model, X, y, epochs, lr=0.01, batch_size=64, seed=0):
    torch.manual_seed(seed)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    Xt, yt = torch.tensor(X), torch.tensor(y)
    for _ in range(epochs):
        order = torch.randperm(len(Xt))
        for i in range(0, len(Xt), batch_size):
            idx = order[i:i + batch_size]
            opt.zero_grad()
            loss_fn(model(Xt[idx]), yt[idx]).backward()
            opt.step()
    return model


def accuracy(model, X, y):
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X)).argmax(dim=1)
    model.train()
    return float((pred == torch.tensor(y)).float().mean())


torch.manual_seed(0)
dense = OneHot(nn.Sequential(nn.Flatten(), nn.Linear(10 * 3, 64), nn.ReLU(),
                             nn.Linear(64, 2)))
train(dense, X10, y10, epochs=20)
print(f"dense network at length 10: {accuracy(dense, X_test10, y_test10):.4f}")

try:
    accuracy(dense, X20, y20)
    print("dense network at length 20: it ran")
except RuntimeError as err:
    print(f"dense network at length 20: RuntimeError: {err}")


That error is the whole argument for this unit. The network does not score badly
on a length it was not built for. It cannot be given one.

## Recurrence: the same weights at every step

A recurrent network reads one step at a time and carries a hidden state forward.
The weights do not depend on the position, so the same network accepts any
length.


In [ ]:
class Recurrent(nn.Module):
    """Embedding, then a recurrent layer, then a linear layer on the last state."""

    def __init__(self, kind, vocab, classes, hidden=32, forget_bias=None):
        super().__init__()
        self.emb = nn.Embedding(vocab, 16)
        self.rnn = (nn.RNN if kind == "rnn" else nn.LSTM)(16, hidden, batch_first=True)
        self.out = nn.Linear(hidden, classes)
        if kind == "lstm" and forget_bias is not None:
            # PyTorch packs the four LSTM gate biases into one vector, in the
            # order input, forget, cell, output. The forget gate is the second
            # quarter, and it is the only one being changed here.
            for name, p in self.rnn.named_parameters():
                if "bias" in name:
                    q = p.shape[0] // 4
                    with torch.no_grad():
                        p[q:2 * q].fill_(forget_bias)

    def forward(self, x):
        out, _ = self.rnn(self.emb(x))
        return self.out(out[:, -1, :])


started = time.time()
for kind in ("rnn", "lstm"):
    torch.manual_seed(0)
    model = Recurrent(kind, vocab=3, classes=2)
    train(model, X10, y10, epochs=20)
    print(f"{kind:4s}  trained at length 10: {accuracy(model, X_test10, y_test10):.4f}"
          f"   tested at length 20: {accuracy(model, X20, y20):.4f}")
print(f"({time.time() - started:.1f}s)")


Both accept the longer input, which the dense network could not, and both lose
accuracy on a length they were never shown. Accepting a length is not the same as
working at it.

## Train each one at the length it is tested on

Give each network the length it will be scored at and the problem disappears.


In [ ]:
started = time.time()
for length in (10, 40, 80):
    row = []
    X_tr, y_tr = order_task(4000, length, 10 + length)
    X_te, y_te = order_task(1000, length, 20 + length)
    for kind in ("rnn", "lstm"):
        torch.manual_seed(0)
        model = Recurrent(kind, vocab=3, classes=2)
        train(model, X_tr, y_tr, epochs=20)
        row.append(f"{kind} {accuracy(model, X_te, y_te):.4f}")
    print(f"length {length:2d}:  " + "   ".join(row))
print(f"({time.time() - started:.1f}s)")


Everything reads 1.0000, including eighty steps deep. That looks like the
architecture coping with length.

It is not. The order task can be solved from whichever of `A` or `B` is seen
**last**, which is near the end of the sequence, so nothing has to survive the
whole trip. The score says the task was solved. It says nothing about where in
the sequence the network was working.

## Measure the gradient instead

The gradient reaching step 1, compared with the gradient reaching the last step.

**This is measured at initialisation, before any training.** Comparing a trained
network that works with a trained network that does not measures how converged
they are, not how well gradient travels: a model stuck at chance has a large loss
and large gradients everywhere. At initialisation both are equally untrained, so
the only thing separating them is the architecture.


In [ ]:
def gradient_profile(model, X, y):
    """Average gradient magnitude arriving at the first and the last input."""
    e = model.emb(torch.tensor(X)).detach().requires_grad_(True)
    out, _ = model.rnn(e)
    nn.CrossEntropyLoss()(model.out(out[:, -1, :]), torch.tensor(y)).backward()
    g = e.grad.abs().mean(dim=(0, 2))
    return float(g[0]), float(g[-1])


X_grad, y_grad = recall_task(512, 80, 99)
print("length 80, at initialisation, median of five seeds\n")
print(f"{'':24s} {'step 1':>11s}  {'last step':>11s}   decay")
for label, kind, bias in [("plain RNN", "rnn", None),
                          ("LSTM, forget bias 0", "lstm", 0.0),
                          ("LSTM, forget bias 1", "lstm", 1.0)]:
    firsts, lasts = [], []
    for seed in range(5):
        torch.manual_seed(seed)
        net = Recurrent(kind, CLASSES + 1 + FILLER, CLASSES, forget_bias=bias)
        a, b = gradient_profile(net, X_grad, y_grad)
        firsts.append(a)
        lasts.append(b)
    f, l = np.median(firsts), np.median(lasts)
    print(f"{label:24s} {f:11.3e}  {l:11.3e}   {l / f:.3e}x")


The plain RNN's first step receives about **4e-26** against **7e-05** at the last
step. That is not a small gradient, it is no gradient: multiplied by any sensible
learning rate it changes the weight by nothing a 32-bit float can represent.

This is lesson 3.4.1 with timesteps in place of layers. Eighty steps of
multiplication does what eighty layers of it does.

**The LSTM with its default forget bias is barely better**, 2e-21 against 4e-26.
Both are zero for any practical purpose, and a ratio between two quantities that
are both effectively zero does not make either one usable.

**The last line is the one that matters.** Setting the forget-gate bias to 1
takes the decay from a factor of 10^15 to a factor of about 88.

## A task that actually needs the first step

The order task hid all of this because it could be solved from the end. The
recall task cannot: the answer is the first token and everything after it is
noise.


In [ ]:
started = time.time()
print("recall task, first of eight, chance 0.125\n")
print("length   plain RNN   LSTM")
for length in (20, 40, 60):
    X_tr, y_tr = recall_task(6000, length, 30 + length)
    X_te, y_te = recall_task(1000, length, 40 + length)
    row = []
    for kind in ("rnn", "lstm"):
        torch.manual_seed(0)
        model = Recurrent(kind, CLASSES + 1 + FILLER, CLASSES)
        train(model, X_tr, y_tr, epochs=40)
        row.append(accuracy(model, X_te, y_te))
    print(f"{length:6d}   {row[0]:.4f}      {row[1]:.4f}")
print(f"({time.time() - started:.1f}s)")


At length 20 both are perfect. At 40 and 60 both are at or near chance, and no
amount of training fixes it, because the information has to survive forty
sequential multiplications and it does not.

Note that the LSTM is not better here. Its default forget bias is 0, which the
gradient table above already showed is barely different from a plain RNN.

## One line that changes it


In [ ]:
started = time.time()
X_tr, y_tr = recall_task(6000, 40, 70)
X_te, y_te = recall_task(1000, 40, 71)

for bias in (0.0, 1.0):
    torch.manual_seed(0)
    model = Recurrent("lstm", CLASSES + 1 + FILLER, CLASSES, forget_bias=bias)
    train(model, X_tr, y_tr, epochs=40)
    print(f"LSTM at length 40, forget bias {bias}: {accuracy(model, X_te, y_te):.4f}")
print(f"({time.time() - started:.1f}s)")


Chance to solved, from one number.

A forget gate near 0 multiplies the carried state by something near zero at every
step, which is the vanishing gradient rebuilt out of gates. Starting that bias at
1 opens the gate, so the default behaviour becomes *keep* rather than *discard*,
and the state survives the trip.

This is the third time this track has met the same arithmetic. Lesson 3.4.1 found
a signal dying through repeated multiplication by layers, lesson 3.11.4 finds it
dying through repeated averaging over neighbours, and here it dies through
repeated multiplication by timesteps. Where the starting value sits decides which
way it goes.

## What to take away

- A recurrent network accepts any length. That is not the same as working at any
  length.
- A score tells you the task was solved. It does not tell you *where* in the
  sequence the network was working, and measuring the gradient does.
- Gates slow the decay; they do not remove it. Sixty steps defeated everything
  here.

The fix was to stop making it sequential: let every position look at every other
position directly, in one step, so nothing has to survive a long chain. That is
attention, and it is where Track 4 begins.
